<a href="https://colab.research.google.com/github/HmanseyWBS/HmanseyWBS---AnalyticsEngineering/blob/main/API/tefl_api_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with the TfL Open API

## Workbook introduction

This workbook introduces the **Transport for London (TfL) Unified API** and shows how to retrieve, inspect, transform, and use live transport data in Python.

TfL publishes a wide range of open datasets and real-time information through its API, covering services including the Underground, buses, Overground, Elizabeth line, trams, river services, Santander Cycles, road disruption data, and journey planning. The API returns information in **JSON** format, which we will request over the web and analyse using Python tools such as `requests` and `pandas`.

The goal is not only to make individual API calls, but to develop a repeatable workflow for working with web APIs:

1. Identify the endpoint and the data it provides.
2. Send a request with the appropriate parameters and credentials.
3. Check that the response was successful and understand its JSON structure.
4. Extract the fields needed for analysis.
5. Convert the results into a structured table or visualisation.
6. Handle missing values, empty results, and request errors safely.

## Learning objectives

By the end of this workbook, you should be able to:

- explain what an API is and why it is useful for accessing live public data;
- make HTTP `GET` requests to the TfL Unified API from Google Colab;
- use query parameters, including an application ID and API key where required;
- inspect and navigate JSON responses;
- create pandas DataFrames from API data;
- retrieve examples of live TfL information, such as service status, arrivals, or cycle-hire availability; and
- adapt an example request to answer a new transport-data question.

## The TfL Unified API

An **Application Programming Interface (API)** is a defined way for one program to request data or services from another. In this workbook, Python is the client: it sends a request to a TfL API endpoint and receives a response containing the requested data.

TfL's base API address is:

```text
https://api.tfl.gov.uk
```

An **endpoint** is the part of the address that identifies a particular service or dataset. For example, an endpoint can return the current status of one or more transport lines. Requests can also include **parameters** to specify exactly what data is needed.

Many TfL endpoints can be explored without authentication. For regular, automated, or higher-volume use, requests should include a TfL `app_id` and `app_key`. These credentials identify the application making the request and help TfL manage fair use of the service. They should not be hard-coded into a shared notebook or committed to a public repository. In Colab, we will store them as secrets and load them only when needed.

## How this workbook will work

Each practical section will follow the same pattern:

- **Question** — define the transport question we want to answer.
- **API request** — construct and run the URL or Python request.
- **Inspect** — examine the response and identify useful fields.
- **Transform** — organise the data into a DataFrame where appropriate.
- **Interpret** — explain what the output tells us and note any limitations.

Live API data changes continuously. Running the same cell at different times can therefore return different arrival predictions, service statuses, availability counts, or disruption details. This is expected and is one of the benefits—and analytical considerations—of using a real-time API.

## Before we begin

This workbook assumes basic familiarity with Python variables, lists, dictionaries, and pandas DataFrames. No previous API experience is required.

As we work, pay attention to three practical questions:

- What does this endpoint provide?
- What is the structure of the returned JSON?
- How can the response be turned into information that answers a useful question?

We will begin by importing the Python libraries needed to send requests and work with returned data.

# 1. Set up the notebook

## What we are doing

Before making an API call, we need to prepare the notebook. We will:

1. import Python libraries for making web requests and working with tabular data;
2. store the TfL API base address in one reusable variable; and
3. make a simple request to check that the API is available.

For this classroom activity, no API registration, `app_id`, or `app_key` is required. We will use publicly accessible TfL API endpoints and keep the focus on making and analysing requests.

## 1.1 Import the libraries

Run the following code cell. `requests` sends HTTP requests to the API, while `pandas` helps us organise returned data into tables.

Google Colab already includes both packages, so no installation should be needed.

In [1]:
import pandas as pd
import requests

# Display all columns when viewing a pandas DataFrame.
pd.set_option("display.max_columns", None)

## 1.2 Define the base URL

All TfL Unified API endpoints begin with the same base address. Store it once so that later requests can add only the endpoint they need.

For example, the URL for the line-status endpoint will later be formed by combining `BASE_URL` with `/Line/Mode/tube/Status`.

In [2]:
BASE_URL = "https://api.tfl.gov.uk"

For example, the URL for the line-status endpoint is formed by combining `BASE_URL` with `/Line/Mode/tube/Status`.

## 1.3 Check that the API is available

For our first request, we will use TfL's line-status endpoint. It returns the current service status for London Underground lines, so it provides both a useful test of the connection and real transport data to inspect.


In [3]:
response = requests.get(
    f"{BASE_URL}/Line/Mode/tube/Status",
    timeout=30
)

print("Status code:", response.status_code)
response.raise_for_status()

line_status = response.json()
print("Response type:", type(line_status).__name__)
print("Lines returned:", len(line_status))

Status code: 200
Response type: list
Lines returned: 11


A status code of `200` means the request was successful. The `response.raise_for_status()` line stops the cell and reports an error if the API returns an unsuccessful response, such as `404` (endpoint not found) or `500` (server error).

## Checkpoint

At this point, the notebook has a reusable API base URL and the tools needed to make public requests to TfL. The next section will make a first meaningful call: retrieving the current status of London Underground lines.
nebulaONE logo

# 2. Inspect and parse the JSON response

## What we are doing

The API has returned data, but it is not yet in the tidy rows-and-columns format used in a spreadsheet or pandas DataFrame. The returned **JSON** is a nested structure made up of lists and dictionaries.

In this section, we will work systematically to:

1. identify the overall type of the response;
2. inspect one item without printing an overwhelming amount of data;
3. identify the keys available at each level;
4. safely access values within nested dictionaries and lists; and
5. extract selected fields into a simple table.

We will continue using the `line_status` variable created in Section 1.3.

## 2.1 Inspect the overall structure

First, check the type and size of the object returned by the API. Then examine the first item in the list.

In [4]:
print("Overall type:", type(line_status).__name__)
print("Number of line records:", len(line_status))

first_line = line_status[0]
print("First item type:", type(first_line).__name__)
print("Top-level keys:")
print(first_line.keys())

Overall type: list
Number of line records: 11
First item type: dict
Top-level keys:
dict_keys(['$type', 'id', 'name', 'modeName', 'disruptions', 'created', 'modified', 'lineStatuses', 'routeSections', 'serviceTypes', 'crowding'])


The result should show that:

- `line_status` is a **list**;
- each element in the list is a **dictionary** representing one Tube line; and
- each dictionary has named keys such as `id`, `name`, and `lineStatuses`.

A list is ordered and uses numeric positions, starting at `0`. A dictionary stores values under named keys. For example, the first line's name can be accessed with `first_line["name"]`.

## 2.2 Look at one record clearly

Printing an entire live API response can make it difficult to see its structure. The `pprint` module displays nested dictionaries and lists in a more readable format.

In [5]:
from pprint import pprint

pprint(first_line)

{'$type': 'Tfl.Api.Presentation.Entities.Line, Tfl.Api.Presentation.Entities',
 'created': '2026-09-22T09:53:41.037Z',
 'crowding': {'$type': 'Tfl.Api.Presentation.Entities.Crowding, '
                       'Tfl.Api.Presentation.Entities'},
 'disruptions': [],
 'id': 'bakerloo',
 'lineStatuses': [{'$type': 'Tfl.Api.Presentation.Entities.LineStatus, '
                            'Tfl.Api.Presentation.Entities',
                   'created': '0001-01-01T00:00:00',
                   'id': 0,
                   'statusSeverity': 10,
                   'statusSeverityDescription': 'Good Service',
                   'validityPeriods': []}],
 'modeName': 'tube',
 'modified': '2026-09-22T09:53:41.037Z',
 'name': 'Bakerloo',
 'routeSections': [],
 'serviceTypes': [{'$type': 'Tfl.Api.Presentation.Entities.LineServiceTypeInfo, '
                            'Tfl.Api.Presentation.Entities',
                   'name': 'Regular',
                   'uri': '/Line/Route?ids=Bakerloo&serviceTypes=Regu

Read the output from the outside inward. The first-line dictionary contains general details about a Tube line. One of its keys, `lineStatuses`, holds another list. That nested list contains the current status record or records for that line.

## 2.3 Access simple values

Extract a few values held directly in the first-line dictionary.


In [6]:
print("Line ID:", first_line["id"])
print("Line name:", first_line["name"])
print("Transport mode:", first_line["modeName"])

Line ID: bakerloo
Line name: Bakerloo
Transport mode: tube


These values are simple because they are stored directly against their keys. The service-status description is nested one level further down, so it needs an additional indexing step.

## 2.4 Navigate nested lists and dictionaries

The `lineStatuses` value is a list. Inspect its type, length, and first item before extracting a value from it.

In [7]:
statuses = first_line["lineStatuses"]

print("Type of lineStatuses:", type(statuses).__name__)
print("Number of status records:", len(statuses))

first_status = statuses[0]
print("Status keys:", first_status.keys())
pprint(first_status)

Type of lineStatuses: list
Number of status records: 1
Status keys: dict_keys(['$type', 'id', 'statusSeverity', 'statusSeverityDescription', 'created', 'validityPeriods'])
{'$type': 'Tfl.Api.Presentation.Entities.LineStatus, '
          'Tfl.Api.Presentation.Entities',
 'created': '0001-01-01T00:00:00',
 'id': 0,
 'statusSeverity': 10,
 'statusSeverityDescription': 'Good Service',
 'validityPeriods': []}


The status description is held in the nested `statusSeverityDescription` key:


In [8]:
print("Current status:", first_status["statusSeverityDescription"])

Current status: Good Service


The full route from the original response to this value is:


In [9]:
line_status[0]["lineStatuses"][0]["statusSeverityDescription"]

'Good Service'

Breaking this into named variables, as we did above, is usually clearer and easier to debug.

## 2.5 Create a tidy line-status table

We can now answer a useful question:

> **What is the current status of each Tube line?**

A line can contain more than one item in its `lineStatuses` list. Therefore, we use a nested loop and create one table row for every status record, rather than assuming that the first item is the only one.


In [10]:
rows = []

for line in line_status:
    for status in line.get("lineStatuses", []):
        rows.append({
            "line_id": line.get("id"),
            "line_name": line.get("name"),
            "mode": line.get("modeName"),
            "status": status.get("statusSeverityDescription"),
            "severity": status.get("statusSeverity"),
            "reason": status.get("reason")
        })

status_df = pd.DataFrame(rows)
status_df

,line_id,line_name,mode,status,severity,reason
0,bakerloo,Bakerloo,tube,Good Service,10,None
1,central,Central,tube,Minor Delays,9,Central Line: Minor delays due to train cancel...
2,circle,Circle,tube,Good Service,10,None
3,district,District,tube,Severe Delays,6,District Line: Severe delays due to an earlier...
4,hammersmith-city,Hammersmith & City,tube,Good Service,10,None
5,jubilee,Jubilee,tube,Good Service,10,None
6,metropolitan,Metropolitan,tube,Good Service,10,None
7,northern,Northern,tube,Good Service,10,None
8,piccadilly,Piccadilly,tube,Good Service,10,None
9,victoria,Victoria,tube,Minor Delays,9,Victoria Line: Minor delays due to train cance...


The **outer loop** processes each Tube line. The **inner loop** processes every status record for that line. Each call to `rows.append()` adds a simple dictionary to the `rows` list, and `pd.DataFrame(rows)` converts that list into a table.

The `reason` field is not always supplied. Using `status.get("reason")` means Python returns `None` when a reason is absent, rather than stopping with a `KeyError`.

## 2.6 Sort and display the table

Sort the records by severity and line name, then reset the row numbering.

In [11]:
status_df = status_df.sort_values(
    by=["severity", "line_name"]
).reset_index(drop=True)

status_df

,line_id,line_name,mode,status,severity,reason
0,district,District,tube,Severe Delays,6,District Line: Severe delays due to an earlier...
1,central,Central,tube,Minor Delays,9,Central Line: Minor delays due to train cancel...
2,victoria,Victoria,tube,Minor Delays,9,Victoria Line: Minor delays due to train cance...
3,bakerloo,Bakerloo,tube,Good Service,10,None
4,circle,Circle,tube,Good Service,10,None
5,hammersmith-city,Hammersmith & City,tube,Good Service,10,None
6,jubilee,Jubilee,tube,Good Service,10,None
7,metropolitan,Metropolitan,tube,Good Service,10,None
8,northern,Northern,tube,Good Service,10,None
9,piccadilly,Piccadilly,tube,Good Service,10,None


In TfL status data, a lower numerical severity generally represents a more serious problem. Sorting by this field makes any disruption records easier to find near the top of the table.

The `reason` column may contain a longer explanation of a disruption. It will often be empty (`None`) for lines running normally.

## 2.7 Identify lines without Good Service

We can now filter the DataFrame to focus on lines whose status is not reported as `Good Service`.

In [12]:
disrupted_lines = status_df[
    status_df["status"] != "Good Service"
]

disrupted_lines

,line_id,line_name,mode,status,severity,reason
0,district,District,tube,Severe Delays,6,District Line: Severe delays due to an earlier...
1,central,Central,tube,Minor Delays,9,Central Line: Minor delays due to train cancel...
2,victoria,Victoria,tube,Minor Delays,9,Victoria Line: Minor delays due to train cance...


If every line is operating normally, this may return an empty DataFrame. That is a valid result: it means that, at the time of the request, TfL reported `Good Service` for all returned lines.

To show a clearer message in that case, run:

In [13]:
if disrupted_lines.empty:
    print("All returned Tube lines have Good Service.")
else:
    display(disrupted_lines)

,line_id,line_name,mode,status,severity,reason
0,district,District,tube,Severe Delays,6,District Line: Severe delays due to an earlier...
1,central,Central,tube,Minor Delays,9,Central Line: Minor delays due to train cancel...
2,victoria,Victoria,tube,Minor Delays,9,Victoria Line: Minor delays due to train cance...


Because the data is live, rerunning the request in Section 1.3 and then the parsing cells may produce different results.

## 2.8 Check assumptions before indexing

The earlier examples use `[0]` to inspect the first line and its first status. This is appropriate here because the request returned records, but it is good practice to check before indexing a list. A different endpoint, filter, or future API response could return an empty list.


In [14]:
if not line_status:
    print("No line-status records were returned.")
else:
    first_line = line_status[0]
    statuses = first_line.get("lineStatuses", [])

    print("First line:", first_line.get("name"))
    print("Status records for this line:", len(statuses))

    if statuses:
        print("First status:", statuses[0].get("statusSeverityDescription"))
    else:
        print("No status records were supplied for this line.")

First line: Bakerloo
Status records for this line: 1
First status: Good Service


This makes two distinctions that are important when using live APIs:

- a key might not be present in a dictionary; and
- a key may be present, but its list may be empty.

The `.get()` method handles a missing dictionary key safely. The `if statuses:` test checks whether a list contains at least one item before attempting to use `statuses[0]`.

## 2.9 Summarise the returned status data

A DataFrame makes it straightforward to move from individual records to a summary. First, count the number of status rows in each category.


In [15]:
status_counts = (
    status_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="number_of_records")
)

status_counts

,status,number_of_records
0,Good Service,8
1,Minor Delays,2
2,Severe Delays,1


`value_counts()` counts repeated values in one column. The result answers a slightly different question from the earlier table:

> **How many returned line-status records belong to each status category?**

We can also calculate the proportion of returned records that are reporting `Good Service`.

In [16]:
good_service_percentage = (
    status_df["status"].eq("Good Service").mean() * 100
)

print(f"Good Service: {good_service_percentage:.1f}% of returned status records")

Good Service: 72.7% of returned status records


Here, `.eq("Good Service")` produces `True` or `False` for each row. In Python and pandas, `True` is treated as `1` and `False` as `0` when calculating a mean, so the mean is the proportion of records with that status.

## 2.10 Explore the data independently

Use the existing `status_df` DataFrame to answer the following questions before looking at the suggested code.

1. How many unique Tube lines were returned?
2. Which fields are missing most often?
3. If there are disrupted lines, what reasons has TfL supplied?

Suggested code:

In [17]:
print("Unique lines:", status_df["line_id"].nunique())

print("\nMissing values by column:")
print(status_df.isna().sum())

print("\nDisruption reasons:")
disruption_reasons = disrupted_lines[
    ["line_name", "status", "reason"]
].dropna(subset=["reason"])

display(disruption_reasons)

Unique lines: 11

Missing values by column:
line_id      0
line_name    0
mode         0
status       0
severity     0
reason       8
dtype: int64

Disruption reasons:


,line_name,status,reason
0,District,Severe Delays,District Line: Severe delays due to an earlier...
1,Central,Minor Delays,Central Line: Minor delays due to train cancel...
2,Victoria,Minor Delays,Victoria Line: Minor delays due to train cance...


`nunique()` counts distinct non-missing values. `isna().sum()` counts missing values in each column. The final block selects the most useful fields for explaining a disruption and removes records where TfL has not supplied a reason.

## Checkpoint

You have now converted nested JSON into a tidy DataFrame, checked for missing or empty data safely, and used the table to identify and summarise Tube service disruptions. The general workflow is:

1. check the response type;
2. inspect one example item;
3. identify lists, dictionaries, and useful keys;
4. navigate nested levels carefully;
5. extract one row per relevant nested record;
6. check assumptions about missing keys and empty lists; and
7. filter or summarise the resulting table to answer a question.

The next endpoint may use a different JSON structure, but this same inspection process should be applied before attempting to analyse it.